# Running circuits with `QBackend`

A QaaS backend is a Qiskit `BackendV2`: `backend.run(...)` returns a `QJob`, and `job.result()` returns a Qiskit `Result` (counts and per-shot memory), as with any Qiskit backend.

For results in the Qiskit primitives format (`PrimitiveResult`), see `sampler_QSamplerV2_qheappe.ipynb`.

In [ ]:
%pip install pylatexenc
%pip install matplotlib
%pip install 'qaas[iqm_qiskit]>=1.1.0' --extra-index-url https://opencode.it4i.eu/api/v4/projects/107/packages/pypi/simple

In [ ]:
import os

os.environ["QPROVIDER_LOGLEVEL"] = "INFO"

# Connect to system

In [ ]:
from qheappe_client import QBackend, QJob, QProvider

lexis_project = "EU-xx-xxx"
resource_name = "VLQ-EU"

provider = QProvider(lexis_project)

## or define manually LEXIS access token e.g. via dotenv

## %pip install python-dotenv
## from dotenv import load_dotenv
## load_dotenv()

## provider = QProvider(LEXIS_PROJECT, token)

In [ ]:
backend: QBackend = provider.get_backend(resource_name, backend_name="VLQ")

print("Backend:", backend.name)
print("Qubits:", len(backend.architecture.qubits))
print("Native gates:", list(backend.architecture.gates.keys()))
print("Default options:", backend.options)  # shots=1024, memory=False

# 1. Run a circuit

Circuits must be transpiled for the backend first. `shots` defaults to `backend.options.shots`.

In [ ]:
from qiskit import QuantumCircuit

bell = QuantumCircuit(2, name="bell")
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

bell_isa = backend.transpile(bell)

job: QJob = backend.run(bell_isa, shots=1000, job_name="bell_state")
print("HEAppE job ID:", job.job_id())

## Waiting for the job

`wait_for_final_state()` follows HEAppE's event stream, so it returns as soon as the job finishes. It's more efficient than calling `job.status()` in a loop. The job info that `status()` reads can still say the job is queued for a few seconds after it finished.

`job.result()` waits the same way, so calling `wait_for_final_state()` first is optional.

In [ ]:
job.wait_for_final_state(
    timeout=600, callback=lambda job_id, status, job: print(job_id, status.name)
)
print("Status:", job.status().name, "| done:", job.done())

## Reading the result

In [ ]:
result = job.result()  # qiskit.result.Result

counts = result.get_counts()
print("Counts:", counts)
print("Counts of this circuit:", result.get_counts(bell_isa))  # look up by circuit

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts, figsize=(8, 4), title="Bell state")

# 2. Per-shot results (memory)

Pass `memory=True` to `run()` to get each shot's bitstring with `result.get_memory()`, as with other Qiskit backends.

In [ ]:
job = backend.run(bell_isa, shots=200, memory=True)
result = job.result()

memory = result.get_memory()
print("First 10 shots:", memory[:10])
print("Shots:", len(memory))

# 3. Several classical registers

Bitstrings follow the Qiskit order: registers in circuit order, separated by a space, and bit 0 rightmost.

In [ ]:
from qiskit import ClassicalRegister, QuantumRegister

a, b = ClassicalRegister(2, "a"), ClassicalRegister(1, "b")
regs = QuantumCircuit(QuantumRegister(3), a, b, name="registers")
regs.x(0)  # expected: a = "01", b = "0", so counts key "0 01"
regs.measure([0, 1], a)
regs.measure(2, b)

result = backend.run(backend.transpile(regs), shots=500).result()
print("Counts:", result.get_counts())
print("Registers:", result.results[0].header["creg_sizes"])

# 4. Several circuits in one job

Pass a list of circuits to run them in one HEAppE job; read each with `result.get_counts(i)`.

In [ ]:
ghz = QuantumCircuit(3, name="ghz")
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(0, 2)
ghz.measure_all()

circuits = backend.transpile([bell, ghz])
result = backend.run(circuits, shots=500).result()

for i, circuit in enumerate(circuits):
    print(f"{circuit.name}: {result.get_counts(i)}")

# 5. Cancelling a job

`cancel()` asks HEAppE to cancel the job and returns whether the request was accepted.

In [ ]:
long_job = backend.run(backend.transpile(ghz), shots=20000, job_name="to_cancel")

if long_job.cancel():
    print("Status:", long_job.status().name)  # CANCELLED
else:
    print("HEAppE did not accept the cancel request; status:", long_job.status().name)

# 6. Results of an earlier job

Re-create a job from its HEAppE job ID, e.g. in a new session. Pass `memory=True` for per-shot results.

A re-created job keeps the classical registers, but its experiments are named `circuit-0`, `circuit-1`, … because the original circuits are not available.

In [ ]:
earlier_id = int(job.job_id)  # replace with the ID of your job

earlier = QJob(backend, earlier_id, memory=True)
result = earlier.result()
print("Counts:", result.get_counts())
print("First 10 shots:", result.get_memory()[:10])

# 7. Qiskit tools that take a backend

Because `job.result()` returns a standard `Result`, Qiskit's generic primitives work on a QaaS backend, e.g. `BackendEstimatorV2` for expectation values.

In [ ]:
from qiskit.primitives import BackendEstimatorV2
from qiskit.quantum_info import SparsePauliOp

bell_no_meas = QuantumCircuit(2)
bell_no_meas.h(0)
bell_no_meas.cx(0, 1)
bell_no_meas_isa = backend.transpile(bell_no_meas)

zz = SparsePauliOp("ZZ").apply_layout(bell_no_meas_isa.layout)
estimate = BackendEstimatorV2(backend=backend).run([(bell_no_meas_isa, zz)]).result()
print("<ZZ> =", estimate[0].data.evs)  # ideally 1 for a Bell state